# 3. Inference Workloads using NRP resources including NVIDIA A10 GPUs and Qualcomm Cloud AI 100 Ultra Cards with Hugging Face Models (Instructor: Mohammad Sada)

This session will be about deploying large language models on NRP resources such as the NVIDIA A10 GPUs and the Qualcomm Cloud AI 100 Ultra cards. This session will include details on the architecture of the AI 100 Ultra, including its eight cards, each with four SoCs, for a total of 32 addressable devices each capable of running models up to approximately 25 billion parameters. Participants will learn how to request an individual SoC device and load open-weight models from Hugging Face repositories. They will execute inference workflows, benchmarking model performance, and exploring options for lightweight fine-tuning.

## AI training using PyTorch example

You can use the yaml files from the repo as is but **replace `<username>` with your own username** (e.g. your NRP or GitHub username) in each YAML so resource names are unique and do not collide. Please download the yaml files using the download raw file button 
<img src="https://github.com/user-attachments/assets/4033a577-e8d3-4909-9773-30ab39cafef3" width="256" style="vertical-align:middle"/>
and edit 

We start by running the training example (resources deploy to namespace `nrp-training-k8s`; create it with `kubectl create namespace nrp-training-k8s` if needed):

```
kubectl apply -n nrp-training-k8s -f yamls/pytorch-training.yaml
```

---
## Option A: NVIDIA GPU Inference Workflows

The following sections demonstrate inference workloads running on **NVIDIA GPUs** (e.g., A10). These workflows use standard CUDA-enabled containers and request GPUs via `nvidia.com/gpu` in Kubernetes resource limits.


In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/pytorch-training.yaml

Check on status:

```
kubectl get pods -n nrp-training-k8s
```

In [ ]:
!kubectl get pods -n nrp-training-k8s

When the job is in run state you can check the logs for output:

```
kubectl logs -n nrp-training-k8s tutorial-<username>-gp3
```

In [ ]:
!kubectl logs -n nrp-training-k8s tutorial-<username>-gp3

Once you are done exploring, please delete the pod:

```
kubectl delete -n nrp-training-k8s -f yamls/pytorch-training.yaml
```

In [ ]:
!kubectl delete -n nrp-training-k8s -f yamls/pytorch-training.yaml

## Text generation inference example

Start up the inference pod:

```
kubectl apply -n nrp-training-k8s -f yamls/tgi-inference.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/tgi-inference.yaml

Once the pod is running, get interactive access to the pod:

```
kubectl exec -it -n nrp-training-k8s tutorial-<username>-tgi -- /bin/bash
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-tgi -- /bin/bash

Once in the pod, start a python3 interpreter and then run:

```
from huggingface_hub import InferenceClient
client = InferenceClient(model="http://0.0.0.0:80")
for token in client.text_generation("Who made cat videos?", max_new_tokens=24, stream=True): print (token)
```

In [ ]:
from huggingface_hub import InferenceClient
client = InferenceClient(model="http://0.0.0.0:80")
for token in client.text_generation("Who made cat videos?", max_new_tokens=24, stream=True): print (token)

## RAG example using Ollama

Start up the pod:
```
kubectl apply -n nrp-training-k8s -f yamls/ollama-rag.yaml
```

In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/ollama-rag.yaml

Watch the logs and make sure you wait till the installs are done and the book is downloaded:

```
kubectl logs -n nrp-training-k8s tutorial-<username>-ollama
```
Replace `<username>` with the value you used in the YAML.

In [ ]:
!kubectl logs -n nrp-training-k8s tutorial-<username>-ollama

Once the book is downloaded (you will see wget output in the logs), we can get interactive access to the pod and start up the Ollama server and pull the module we want to use (Mistral):

```
kubectl exec -it ollama-username -- /bin/bash
cd /scratch
nohup ollama serve&
ollama pull mistral
```

In [ ]:
!kubectl exec -it -n nrp-training-k8s tutorial-<username>-ollama -- /bin/bash
!cd /scratch
!nohup ollama serve&
!ollama pull mistral

We can now download our test script and run it:
```
wget https://raw.githubusercontent.com/nrp-nautilus/nairr-tutorial/main/scripts/test.py
python3 -i test.py
```

In [ ]:
!wget https://raw.githubusercontent.com/nrp-nautilus/nairr-tutorial/main/scripts/test.py
!python3 -i test.py

Now we can run the rag within the interactive python interpreter. Do the following one by one (i.e. wait for results before moving to the next one)
```
rag.invoke("What do you feed pigeons ?")
rag.invoke("Do tame pigeons have better plumage ?")
rag.invoke("What affects pigeon plumage ?")
```

In [ ]:
rag.invoke("What do you feed pigeons ?")
rag.invoke("Do tame pigeons have better plumage ?")
rag.invoke("What affects pigeon plumage ?")

## Helm based deployment of LLM as service (H2O) 

Install helm. Details at: <https://github.com/helm/helm#install>. Quickest option mignt be to download and use static binaries from the release page referenced in link above.

We will be using the H20 project (<https://github.com/h2oai>). Clone the H2O repository:

```
git clone https://github.com/h2oai/h2ogpt.git
```

In [ ]:
!git clone https://github.com/h2oai/h2ogpt.git

Copy the values file to the h2o directory:

```
cp yamls/h2o-values.yaml h2ogpt
cd h2ogpt
```

In [ ]:
!cp yamls/h2o-values.yaml h2ogpt
!cd h2ogpt

Install the helm chart. Make sure you use a unique name (change username below):

```
helm install h2ogpt-username helm/h2ogpt-chart -f yamls/h2o-values.yaml
```

In [ ]:
!helm install h2ogpt-username helm/h2ogpt-chart -f yamls/h2o-values.yaml

Check the pods (kubectl get pods, maybe grep your username since there will be a lot of pods running) to make sure they are running. Once the pod is running, check the logs and see if h2o is running.

It will take some time to load the model into GPU memory.

We can do port-forward to the pod by running in the terminal:

```
kubectl port-forward h2ogpt-<username>-<hash> 5000:5000 7860:7860
```

In [ ]:
!kubectl port-forward h2ogpt-<username>-<hash> 5000:5000 7860:7860

While the port-forward is running, open another shell and pull the list of models (you need curl installed):

```
curl http://localhost:5000/v1/models
```

In [ ]:
!curl http://localhost:5000/v1/models

If you don't have curl installed, open the [http://localhost:5000/v1/models](http://localhost:5000/v1/models) URL in the browser.

Now we can run a query to the model:

```
curl -X POST "http://localhost:5000/v1/chat/completions" \
-H "Content-Type: application/json" \
-d '{
  "model": "h2oai/h2o-danube2-1.8b-chat",
  "messages": [
          {"role": "user", "content": "How tall are penguins?"}
  ]
}'
```

In [ ]:
!curl -X POST "http://localhost:5000/v1/chat/completions" \
!-H "Content-Type: application/json" \
!-d '{
!  "model": "h2oai/h2o-danube2-1.8b-chat",
!  "messages": [
!          {"role": "user", "content": "How tall are penguins?"}
!  ]
!}'

This helm chart also installs a chat application.

To see the chat UI through the port-forwarding, open [http://localhost:7860](http://localhost:7860) in browser.

Now you can expose the llm by using an ingress (carefully edit all fields with username):

```yaml
apiVersion: networking.k8s.io/v1
kind: Ingress
metadata:
  annotations:
    kubernetes.io/ingress.class: haproxy
  name: llm-test-username
spec:
  rules:
  - host: llm-test-username.nrp-nautilus.io
    http:
      paths:
      - backend:
          service:
            name: h2ogpt-username-web
            port:
              number: 80
        path: /
        pathType: ImplementationSpecific
  tls:
  - hosts:
    - llm-test-username.nrp-nautilus.io
```

You can open llm-test-username.nrp-nautilus.io (with username replaced) in your browser once the forward works. This will give you the H2O interface for chat, ingesting docs etc. Once you are done you can release the resources by doing:

```
helm uninstall h2ogpt-username
```

In [ ]:
!helm uninstall h2ogpt-username

## End

**Please make sure you did not leave any running pods. Jobs and associated completed pods are OK.**

---
## Option B: Qualcomm Cloud AI 100 Ultra Inference Workflows

The following sections demonstrate inference workloads running on **Qualcomm Cloud AI 100 Ultra** accelerators. These devices are exposed in Kubernetes as `qualcomm.com/qaic` and use the SDK container image `ghcr.io/quic/cloud_ai_inference_ubuntu24:1.20.6.0`.

You have two options for running Qualcomm inference:
1. **Via JupyterHub** — Select the NRP Cloud AI Inference profile from the spawner (described below).
2. **Via kubectl** — Deploy a standalone pod using `yamls/qaic-inference.yaml` (described after).


## Qualcomm Cloud AI 100 Ultra Inference via JupyterHub

The NRP provides access to Qualcomm Cloud AI 100 Ultra devices directly through custom JupyterHub profiles. This allows you to launch an isolated environment with the Qualcomm Platform and Apps SDKs pre-installed, perfect for executing optimized inference workflows.

### 1. Launching the Qualcomm Profile
1. Navigate to the [NRP Training JupyterHub](https://training.nrp-nautilus.io).
2. To utilize Qualcomm inference accelerators, select the **NRP Cloud AI Inference** option from the Spawner Profile list.
3. In the form, locate the **Qualcomm Cloud AI Devices** picker and select exactly **1** device.
4. Click **Start** to spawn your instance. Your pod will execute as the `jovyan` user (UID 1000) with a dedicated `50Gi` persistent volume mounted.

### 2. Accessing the Qualcomm Environment
Once your JupyterLab spins up, open a **Terminal** from the launcher. When you start the terminal, you'll see a welcoming message confirming the Qualcomm SDK versions:

```bash
==================================
== Qualcomm Cloud AI Containers ==
==================================

Platform SDK version: 1.20.6.0
Apps SDK version: 1.20.6.0
```

### 3. Activating the Pre-Built vLLM Environment
The Qualcomm image comes bundled with an optimized Python virtual environment that contains the vLLM bindings compiled against the QAIC architecture.

Source the environment in your bash profile or terminal:
```bash
jovyan@jupyter-mfsada-40ucsd-2eedu:~$ source /opt/vllm-env/bin/activate
(vllm-env) jovyan@jupyter-mfsada-40ucsd-2eedu:~$
```
If you are executing from this notebook, we can invoke python directly using the virtual environment's executable path.

### 4. Running Example Inference
The system has built-in offline inference examples structurally mapped under `/opt/qti-aic/integrations/vllm/examples/offline_inference/`. You can navigate here to run pre-compiled model workflows utilizing your requested Qualcomm hardware.

```bash
cd /opt/qti-aic/integrations/vllm/examples/offline_inference/
python qaic.py 
```

**Note:** `qaic.py` executes a `TinyLlama/TinyLlama-1.1B-Chat-v1.0` model using the V0 LLM engine. The script automatically identifies the `qaic` platform, discovers your assigned accelerator at `/dev/accel/`, and manages KV-cache datatyping inherently.
We can execute this natively from the notebook cell below by wrapping the commands with `!`: 


In [ ]:
!(source /opt/vllm-env/bin/activate && cd /opt/qti-aic/integrations/vllm/examples/offline_inference/ && python qaic.py)


### Qualcomm Inference via kubectl

You can also deploy a Qualcomm inference pod directly using `kubectl`, similar to how the NVIDIA TGI pod is deployed above. This approach does not require JupyterHub.

The pod spec requests `qualcomm.com/qaic: 1` and runs the vLLM offline inference example automatically:

```bash
kubectl apply -n nrp-training-k8s -f yamls/qaic-inference.yaml
```


In [ ]:
!kubectl apply -n nrp-training-k8s -f yamls/qaic-inference.yaml

Check the pod status and logs:
```bash
kubectl get pods
kubectl logs -n nrp-training-k8s tutorial-<username>-qaic-inference
```


In [ ]:
!kubectl get pods
!kubectl logs -n nrp-training-k8s tutorial-<username>-qaic-inference

Once done, clean up:
```bash
kubectl delete -n nrp-training-k8s -f yamls/qaic-inference.yaml
```


In [ ]:
!kubectl delete -n nrp-training-k8s -f yamls/qaic-inference.yaml